In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib import colors as cols
import numpy as np
from lib import vaspwfc
from pymatgen.core import Lattice, Structure, Molecule
from pymatgen.electronic_structure.core import Spin, Orbital
from pymatgen.io.ase import Atoms, AseAtomsAdaptor
from pymatgen.io import vasp
import nglview as nv
from ase.neighborlist import natural_cutoffs, NeighborList
import h5py
from math import pi, exp, sin, cos, log

In [3]:
class Initio:
    def __init__(self):
        pass

    def read_vasp_file(self, path: str) -> vaspwfc | Structure | bool:
        base_name = os.path.basename(path)
        
        match base_name:
            case "WAVECAR":
                wfc = self.get_wavecar(path, lgamma = False)
                return wfc
            case "POSCAR" | "CONTCAR":
                struc = self.get_structure(path)
                return struc
            case _:
                print(f"Could not determine file type of provided file {path}")
                return False

    def get_wavecar(self, path: str, lgamma: bool = True) -> vaspwfc:
        try:
            wfc = vaspwfc(path, lgamma = lgamma)
            return wfc
        except Exception as e:
            print("Error loading the wavecar")
            return False

    def get_structure(self, path: str) -> Structure:
        try:
            structure = Structure.from_file(path)
            return structure
        except Exception as e:
            print("Error loading the structure")
            return False

    def DOS_from_energies(self, eigenenergies: list | np.ndarray = [], gamma = None, sigma = None, energy_range = None, points = None, dE: float = 0.1, weights: list | np.ndarray = []) -> np.ndarray:
        use_weights = False
        
        if isinstance(eigenenergies, list): eigenenergies = np.array(eigenenergies, dtype = float)
        if not isinstance(eigenenergies, np.ndarray): raise TypeError("No valid energy list given")
        
        if isinstance(weights, list): weights = np.array(weights, dtype = float)
        if isinstance(weights, np.ndarray) and len(weights) == len(eigenenergies): use_weights = True

        E_min = np.min(eigenenergies)
        E_max = np.max(eigenenergies)
        
        if isinstance(energy_range, list | np.ndarray):
            energy_range.sort()
            if len(energy_range) > 1:
                E_min = energy_range[0]
                E_max = energy_range[1]
        
        if isinstance(points, int): # Explicit specification of the number of points triggers the energy list to be composed using linspace
            E_list = np.linspace(E_min, E_max, points, dtype = float)
        else: # Use energy spacing instead
            E_list = np.arange(E_min, E_max + dE, dE)
            
        DOS = np.stack([E_list, np.zeros_like(E_list)], dtype = float)
        


        # Use Lorentzian broadening
        if isinstance(gamma, float) and gamma > 0:
            gamma2 = gamma ** 2
            
            for index, energy in enumerate(E_list):
                en_diff_list = eigenenergies - energy
                en_diff_list2 = en_diff_list ** 2
                
                if use_weights:
                    for eigenenergy_index, delta_E2 in enumerate(en_diff_list2):
                        DOS[1, index] += weights[eigenenergy_index] * gamma / (gamma2 + delta_E2)
                else:
                    for delta_E2 in en_diff_list2:
                        DOS[1, index] += gamma / (gamma2 + delta_E2)
        
        return DOS

    def get_HOMO_LUMO(self, wavecar_object: vaspwfc) -> dict:
        all_bands = wavecar_object._bands
        all_band_occs = wavecar_object._occs
        
        [bands_up, bands_down] = [all_bands[i] for i in range(2)]
        [occs_up, occs_down] = [all_band_occs[i] for i in range(2)]
        
        [bands_up_gamma, bands_down_gamma] = [bands_up[0], bands_down[0]] # Assuming k-point 0 is the Gamma point
        [occs_up_gamma, occs_down_gamma] = [occs_up[0], occs_down[0]]
        
        LUMO_up_index = int(np.where(occs_up_gamma < .5)[0][0])
        HOMO_up_index = LUMO_up_index - 1
        LUMO_down_index = int(np.where(occs_down_gamma < .5)[0][0])
        HOMO_down_index = LUMO_down_index - 1
        
        HOMO_up_energy = float(bands_up_gamma[HOMO_up_index])
        HOMO_down_energy = float(bands_down_gamma[HOMO_down_index])
        LUMO_up_energy = float(bands_up_gamma[LUMO_up_index])
        LUMO_down_energy = float(bands_down_gamma[LUMO_down_index])
        
        return {"HOMO_up_index": HOMO_up_index, "HOMO_down_index": HOMO_down_index, "LUMO_up_index": LUMO_up_index, "LUMO_down_index": LUMO_down_index,
                "HOMO_up_energy": HOMO_up_energy, "HOMO_down_energy": HOMO_down_energy, "LUMO_up_energy": LUMO_up_energy, "LUMO_down_energy": LUMO_down_energy}

    def get_eigenenergies_from_wavecar(self, wavecar_object: vaspwfc) -> dict:
        all_bands = wavecar_object._bands
        all_band_occs = wavecar_object._occs
        
        [bands_up, bands_down] = [all_bands[i] for i in range(2)]
        [occs_up, occs_down] = [all_band_occs[i] for i in range(2)]
        
        [bands_up_gamma, bands_down_gamma] = [bands_up[0], bands_down[0]] # Assuming k-point 0 is the Gamma point
        [occs_up_gamma, occs_down_gamma] = [occs_up[0], occs_down[0]]
        
        return {"energies": {"spin up": bands_up_gamma, "spin down": bands_down_gamma}, "occupations": {"spin up": occs_up_gamma, "spin down": occs_down_gamma}}

    def spin_and_occupation_resolved_DOS(self, wavecar_object: vaspwfc, *args, **kwargs) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        weights = kwargs.pop("weights", None)
        
        energy_dict = self.get_eigenenergies_from_wavecar(wavecar_object)
        [bands_up_gamma, bands_down_gamma] = [energy_dict["energies"][spin] for spin in ["spin up", "spin down"]]
        [occs_up_gamma, occs_down_gamma] = [energy_dict["occupations"][spin] for spin in ["spin up", "spin down"]]

        LDOS_up_occ = self.DOS_from_energies(bands_up_gamma, weights = occs_up_gamma, *args, **kwargs)
        LDOS_up_unocc = self.DOS_from_energies(bands_up_gamma, weights = 1 - occs_up_gamma, *args, **kwargs)
        LDOS_down_occ = self.DOS_from_energies(bands_down_gamma, weights = occs_down_gamma, *args, **kwargs)
        LDOS_down_unocc = self.DOS_from_energies(bands_down_gamma, weights = 1 - occs_down_gamma, *args, **kwargs)
        
        return (LDOS_up_occ, LDOS_up_unocc, LDOS_down_occ, LDOS_down_unocc)

    def DOS_plot(self, wavecar_object: vaspwfc, *args, **kwargs) -> plt.Figure:
        colors = kwargs.pop("colors", None)
        
        # No colors given. Use defaults
        if not isinstance(colors, list) or len(colors) < 2: colors = ["#A00000", "#0000A0"]
        # Invalid colors given. Use defaults
        if not cols.is_color_like(colors[0]): colors = ["#A00000", "#0000A0"]

        col_up_occ = list(cols.to_rgb(colors[0]))
        col_up_unocc = [.5 + .5 * channel for channel in col_up_occ]
        col_down_occ = list(cols.to_rgb(colors[1]))
        col_down_unocc = [.5 + .5 * channel for channel in col_down_occ]
        
        (LDOS_up_occ, LDOS_up_unocc, LDOS_down_occ, LDOS_down_unocc) = self.spin_and_occupation_resolved_DOS(wavecar_object, *args, **kwargs)
        
        fig, ax = plt.subplots()
        ax.fill_betweenx(LDOS_up_occ[0], LDOS_up_occ[1], color = col_up_occ)
        ax.fill_betweenx(LDOS_up_unocc[0], LDOS_up_unocc[1], color = col_up_unocc)
        ax.fill_betweenx(LDOS_down_occ[0], -LDOS_down_occ[1], color = col_down_occ)
        ax.fill_betweenx(LDOS_down_unocc[0], -LDOS_down_unocc[1], color = col_down_unocc)
        
        ax.set_xlabel("total DOS up    total DOS down")
        ax.set_ylabel("energy (eV)")
        
        en_range = kwargs.get("energy_range")
        ax.set_ylim(en_range[0], en_range[1])
        ax.yaxis.set_minor_locator(ticker.MultipleLocator(.1))
        
        ax.grid(True, which = "both", axis = "y", color = "gray", linewidth = 0.5, alpha = 0.5)
        return fig



In [4]:
WS2_folder = "C:\\DFT\\WS2"

calculation_folders = {name: os.path.join(WS2_folder, name) for name in ["primitive", "Vac_S_0", "supercell", "Co_S_0", "Co_S_-1", "Co_S_+1"]}

In [249]:
calculation_name = "Co_S_-1"
os.chdir(calculation_folders[calculation_name])
print(f"Analyzing files in calculation folder {os.getcwd()}")

initio = Initio()
[struc, wfc] = [initio.read_vasp_file(file_name) for file_name in ["./CONTCAR", "./WAVECAR"]]

Analyzing files in calculation folder C:\DFT\WS2\Co_S_-1


In [ ]:
HL_dict = initio.get_HOMO_LUMO(wfc)
[HOMO_up_index, HOMO_down_index] = [HL_dict.get(attribute) for attribute in ["HOMO_up_index", "HOMO_down_index"]]
print(f"Spin up HOMO / LUMO band indices: [{HOMO_up_index}, {HOMO_up_index + 1}]\nSpin down HOMO / LUMO band indices: [{HOMO_down_index}, {HOMO_down_index + 1}]")

gamma = .03
figure = initio.DOS_plot(wfc, energy_range = [-3.6, 0], dE = .01, gamma = gamma)
plt.tight_layout()
plt.savefig(f"./{calculation_name}_DOS_gamma_{int(1000 * gamma)}_meV.svg")
plt.show()

In [243]:
cell_size_Ang = wfc._Acell
cell_size_nm = cell_size_Ang * .1
frame_dict = {"domain (nm)": np.array([cell_size_nm[0, 0], cell_size_nm[1, 1], cell_size_nm[2, 2]], dtype = np.float32)}
w_nm = frame_dict["domain (nm)"][0]
h_nm = frame_dict["domain (nm)"][1]
z_nm = frame_dict["domain (nm)"][2]

voxels = wfc._ngrid * 2
spin_index = 2
k_index = 1

x_values = np.linspace(0, w_nm, voxels[0])
y_values = np.linspace(0, h_nm, voxels[1])
z_values = np.linspace(0, z_nm, voxels[2])
orbital_indices = np.arange(HOMO_up_index - 16, HOMO_up_index + 15)
spins = np.array(["spin up", "spin down"], dtype = np.str_)

energy_dict = initio.get_eigenenergies_from_wavecar(wfc)
spin_up_energies = energy_dict["energies"]["spin up"][orbital_indices]
spin_down_energies = energy_dict["energies"]["spin down"][orbital_indices]
spin_up_occupations = energy_dict["occupations"]["spin up"][orbital_indices]
spin_down_occupations = energy_dict["occupations"]["spin down"][orbital_indices]

wfns = np.zeros((len(spins), len(orbital_indices), len(x_values), len(y_values), len(z_values)), dtype = np.complex64)
for spin_index, spin_name in enumerate(spins):
    for index, orb_index in enumerate(orbital_indices):
        wfns[spin_index, index] = wfc.wfc_r(spin_index + 1, k_index, orb_index)

string_dt = h5py.string_dtype(encoding='utf-8')
with h5py.File(f"./{calculation_name}_wfns.h5", "w") as f:
    frame_group = f.create_group("frame")
    frame_group.attrs.update({key: value for key, value in frame_dict.items()})
    
    main_group = f.create_group("wavefunctions")
    main_group.create_dataset("wavefunctions", data = wfns, dtype = np.complex64)    
    main_group.create_dataset("spin", data = np.array([0, 1], dtype = np.uint32), dtype = np.uint32)
    main_group.create_dataset("orbital", data = orbital_indices, dtype = np.uint32)
    main_group.create_dataset("x (nm)", data = x_values, dtype = np.float32)
    main_group.create_dataset("y (nm)", data = y_values, dtype = np.float32)
    main_group.create_dataset("z (nm)", data = z_values, dtype = np.float32)
    
    main_group.attrs.update({"NX_class": "NXdata", "signal": "wavefunctions", "axes": ["spin", "orbital", "x (nm)", "y (nm)", "z (nm)"]})
    
    lattice_group = f.create_group("lattice")
    lattice_group.create_dataset("lattice_vectors (nm)", data = (wfc._Acell) * .1, dtype = np.float32)
    lattice_group.create_dataset("reciprocal_vectors (nm-1)", data = (wfc._Bcell) * 10, dtype = np.float32)
    lattice_group.create_dataset("k_path (nm-1)", data = (wfc._kpath) * 10, dtype = np.float32)
    
    energy_group = f.create_group("energy")
    energy_group.create_dataset("orbital_energies", data = np.stack([spin_up_energies, spin_down_energies]))
    energy_group.create_dataset("orbital_occupations", data = np.stack([spin_up_occupations, spin_down_occupations]))
    energy_group.create_dataset("spin", data = np.array([0, 1], dtype = np.uint32), dtype = np.uint32)


In [244]:
z_slice_height = 0.2

atom_z_values_nm = struc.cart_coords[:, 2] * .1
z_surface = np.mean(atom_z_values_nm[-10:])
z_nm_per_vox = z_nm / voxels[2]

z_target = z_surface + z_slice_height
z_slice_index = int(z_target / z_nm_per_vox)

with h5py.File(f"./{calculation_name}_wfns@{int(z_slice_height * 10)}A.h5", "w") as f:
    frame_group = f.create_group("frame")
    frame_group.attrs.update({key: value for key, value in frame_dict.items()})
    
    main_group = f.create_group("wavefunctions")
    main_group.create_dataset("wavefunctions", data = wfns[:, :, :, :, z_slice_index], dtype = np.complex64)    
    main_group.create_dataset("spin", data = np.array([0, 1], dtype = np.uint32), dtype = np.uint32)
    main_group.create_dataset("orbital", data = orbital_indices, dtype = np.uint32)
    main_group.create_dataset("x (nm)", data = x_values, dtype = np.float32)
    main_group.create_dataset("y (nm)", data = y_values, dtype = np.float32)
    
    main_group.attrs.update({"NX_class": "NXdata", "signal": "wavefunctions", "axes": ["spin", "orbital", "x (nm)", "y (nm)"]})
    
    lattice_group = f.create_group("lattice")
    lattice_group.create_dataset("lattice_vectors (nm)", data = (wfc._Acell) * .1, dtype = np.float32)
    lattice_group.create_dataset("reciprocal_vectors (nm-1)", data = (wfc._Bcell) * 10, dtype = np.float32)
    lattice_group.create_dataset("k_path (nm-1)", data = (wfc._kpath) * 10, dtype = np.float32)
    
    energy_group = f.create_group("energy")
    energy_group.create_dataset("orbital_energies", data = np.stack([spin_up_energies, spin_down_energies]))
    energy_group.create_dataset("orbital_occupations", data = np.stack([spin_up_occupations, spin_down_occupations]))
    energy_group.create_dataset("spin", data = np.array([0, 1], dtype = np.uint32), dtype = np.uint32)

In [245]:
spin_names = ["spin up", "spin down"]

for spin_index, spin_name in enumerate(spin_names):
    for array_index, orbital_index in enumerate(orbital_indices):
        wfn2D = wfns[spin_index, array_index, :, :, z_slice_index]
        density = np.abs(wfn2D) ** 2
        norm_density = density / np.max(density)
        phase = (np.angle(wfn2D) - .25 * pi) % (2 * pi)
        norm_phase = phase / (2 * pi)
        
        orbital_energy = energy_dict["energies"][spin_name][orbital_index]
        
        rgb_img = cols.hsv_to_rgb(np.dstack([norm_phase, np.ones_like(phase), norm_density]))
        #plt.imshow(rgb_img)
        #plt.tight_layout()
        plt.imsave(f"./{calculation_name}_{spin_name}_orb{orbital_index}@{int(z_slice_height * 10)}A.png", rgb_img)


In [246]:
orbital_indices = np.arange(HOMO_up_index - 31, HOMO_up_index + 32)

wfns = np.zeros((len(spins), len(orbital_indices), len(x_values), len(y_values), len(z_values)), dtype = np.complex64)
for spin_index, spin_name in enumerate(spins):
    for index, orb_index in enumerate(orbital_indices):
        wfns[spin_index, index] = wfc.wfc_r(spin_index + 1, k_index, orb_index)

wfns2D = wfns[:, :, :, :, z_slice_index]
densities = np.vstack([np.abs(wfns2D[0]) ** 2, np.abs(wfns2D[0]) ** 2], dtype = np.float32)
energies = np.concatenate([energy_dict["energies"][spin_name][orbital_indices] for spin_name in ["spin up", "spin down"]])
gamma2 = gamma ** 2

In [248]:
for target_energy in np.arange(-3.4, .2, .1):
    en_differences = np.array(energies, dtype = np.float32) - target_energy
    
    weights = gamma / (gamma2 + en_differences ** 2)
    weights /= np.sum(weights)

    image = np.average(densities, axis = 0, weights = weights)
    
    #plt.imshow(image, cmap = "gray")
    #plt.tight_layout()
    plt.imsave(f"./{calculation_name}_LDOS@{int(target_energy * 1000)}meV_{int(z_slice_height * 10)}A.png", image, cmap = "gray")
